Batarya Ömür Tahmini Projesi (Battery Aging Prediction Project)

json Dosyalarını Açma ve İnterpolasyon Yapılmış Verileri Kaydetme
(Decoding json Files and Saving Interpolated Values)

-Veriyi açıp, içinden interpolasyon verilerini çektim 
(Examined the data and getting the interpolated discharge values)

1. Gerekli kütüphaneleri koda ekledim (Added essential libraries to the code)

In [2]:
#Got the essential libraries
import os
import sys
import numpy as np
import json
import h5py
import gc
from tqdm import tqdm

2. Bütün json dosyalarını açtım ve interpolasyon yapılmış verili sıkıştırılmış h5 dosyasına kaydettim
(Opened all json files and saved the interpolated values in an compressed h5 file)

In [3]:
#Listed json files and created the h5 file
json_files = [f for f in os.listdir("FastCharge") if f.endswith(".json")]
with h5py.File("FastCharge_Interpolated_Data.h5", "w") as h5f:

    #Defined the loop for each json file
    for i,json_file in enumerate(tqdm(json_files)): 
        json_path = os.path.join("FastCharge",json_file)
        
        #Opened the json file
        with open(json_path, encoding='utf-8') as f:
            data = json.load(f)
        
        #Extracted the charge protocol to add to the battery's name
        p = data["protocol"]
        protocol = p.rpartition("\\")[2]
        protocol_no_file = protocol.rpartition(".")[0]
        protocol_no_date = protocol_no_file.partition("-")[2]
        protocol_lower_case = protocol_no_date.lower()

        #Searched the interpolated values from the file
        interpolated_values = data.get("cycles_interpolated",{})
        
        #Created a section in h5 file for interpolated values
        if interpolated_values:
            battery_name = os.path.splitext(json_file)[0] + "." + protocol_lower_case
            battery_group = h5f.create_group(battery_name)

            #Examined each numerical value matris in interpolated values and transforming values to float32 format
            for key,flat_list in interpolated_values.items():
                try:
                    np_array = np.array(flat_list,dtype=np.float32)
                except ValueError:
                    del flat_list
                    continue

                #Reshaped matrises to 1000 values for each cycle to analyse easily
                cycles = len(np_array) // 1000
                if cycles > 0:
                    reshaped_matrix = np_array[:cycles * 1000].reshape(cycles,1000)
                    
                    #Saved the values in h5 file
                    battery_group.create_dataset(
                        name=key, 
                        data=reshaped_matrix, 
                        compression="gzip",      
                        compression_opts=4       
                    )
                #Deleted the values saved in h5 file for clean memory usage
                del flat_list
                del np_array
                del reshaped_matrix
        
        #Deleted the saved battery data for clean memory usage
        del f
        del data
        del interpolated_values
        gc.collect()


  0%|          | 0/139 [00:00<?, ?it/s]

100%|██████████| 139/139 [08:06<00:00,  3.50s/it]
